In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from glob import glob

pd.set_option("display.max_columns", None)

# Load Existing Datasets

In [2]:
wbpcb = pd.read_parquet(
    "../../data/processed/wbpcb_master.parquet"
)

metadata = pd.read_csv(
    "../../data/metadata/station_metadata.csv",
    skipinitialspace=True
)

wbpcb["datetime"] = pd.to_datetime(
    wbpcb["datetime"]
)

print(wbpcb.shape)

wbpcb.head()

(508116, 15)


,datetime,station_id,station_name,district,latitude,longitude,aqi,pm25,pm10,humidity,temperature,wind_direction,wind_speed_avg,wind_speed_min,wind_speed_max
0,2022-01-01 21:00:00,WB005,"East Calcutta Girls College, Lake Town",Kolkata,22.601583,88.404556,123,66.79,109.32,71.50,18.09,0,0.0,0.0,0.0
1,2022-01-01 21:00:00,WB009,Leather Complex,Kolkata,22.495300,88.509293,193,87.98,173.04,72.72,18.48,0,0.0,0.0,0.0
2,2022-01-01 21:00:00,WB015,Sarsuna College,Kolkata,22.481270,88.284554,185,85.59,194.13,73.95,18.51,0,0.0,0.0,0.0
3,2022-01-01 22:00:00,WB005,"East Calcutta Girls College, Lake Town",Kolkata,22.601583,88.404556,145,73.56,156.15,75.74,17.73,0,0.0,0.0,0.0
4,2022-01-01 22:00:00,WB009,Leather Complex,Kolkata,22.495300,88.509293,231,99.37,168.51,75.08,18.05,0,0.0,0.0,0.0


# Discover Weather Files

In [3]:
weather_files = sorted(
    glob("../../data/raw/open-meteo-weather/open-meteo*.xlsx")
)

print(
    "Weather Files:",
    len(weather_files)
)

for f in weather_files[:5]:
    print(Path(f).name)

Weather Files: 17
open-meteo-22.46N88.32E10m_VIVEKANANDA COLLEGE FOR WOMEN, BARISHA.xlsx
open-meteo-22.46N88.32E3m_Sarsuna College.xlsx
open-meteo-22.46N88.41E3m_Avidipta Housing Complex.xlsx
open-meteo-22.46N88.51E3m_Leather Complex.xlsx
open-meteo-22.53N88.32E11m_Lorreto College.xlsx


# Clean Weather Column Names

In [4]:
def clean_weather_columns(df):

    rename_map = {
        "temperature_2m (°C)": "temperature_2m",
        "relative_humidity_2m (%)": "relative_humidity_2m",
        "dew_point_2m (°C)": "dew_point_2m",
        "precipitation (mm)": "precipitation",
        "surface_pressure (hPa)": "surface_pressure",
        "cloud_cover (%)": "cloud_cover",
        "wind_speed_10m (km/h)": "wind_speed_10m",
        "wind_direction_10m (°)": "wind_direction_10m",
        "wind_gusts_10m (km/h)": "wind_gusts_10m"
    }

    df = df.rename(columns=rename_map)

    return df

# Schema Validation

In [5]:
required_columns = [
    "time",
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m"
]

for file in weather_files:

    df = pd.read_excel(file)

    df = clean_weather_columns(df)

    missing = (
        set(required_columns)
        - set(df.columns)
    )

    if missing:
        print(
            f"\nERROR: {Path(file).name}"
        )
        print("Missing:", missing)

# Validate Every Weather File

In [6]:
required_columns = [
    "time",
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m"
]

for file in weather_files:

    df = pd.read_excel(file)

    missing = set(required_columns) - set(df.columns)

    if missing:
        print(
            f"\nERROR: {Path(file).name}"
        )
        print("Missing:", missing)


ERROR: open-meteo-22.46N88.32E10m_VIVEKANANDA COLLEGE FOR WOMEN, BARISHA.xlsx
Missing: {'temperature_2m', 'wind_gusts_10m', 'wind_speed_10m', 'dew_point_2m', 'wind_direction_10m', 'precipitation', 'relative_humidity_2m', 'cloud_cover', 'surface_pressure'}

ERROR: open-meteo-22.46N88.32E3m_Sarsuna College.xlsx
Missing: {'temperature_2m', 'wind_gusts_10m', 'wind_speed_10m', 'dew_point_2m', 'wind_direction_10m', 'precipitation', 'relative_humidity_2m', 'cloud_cover', 'surface_pressure'}

ERROR: open-meteo-22.46N88.41E3m_Avidipta Housing Complex.xlsx
Missing: {'temperature_2m', 'wind_gusts_10m', 'wind_speed_10m', 'dew_point_2m', 'wind_direction_10m', 'precipitation', 'relative_humidity_2m', 'cloud_cover', 'surface_pressure'}

ERROR: open-meteo-22.46N88.51E3m_Leather Complex.xlsx
Missing: {'temperature_2m', 'wind_gusts_10m', 'wind_speed_10m', 'dew_point_2m', 'wind_direction_10m', 'precipitation', 'relative_humidity_2m', 'cloud_cover', 'surface_pressure'}

ERROR: open-meteo-22.53N88.32E11m_

# Date Range Validation

In [7]:
for file in weather_files:

    df = pd.read_excel(file)

    df = clean_weather_columns(df)

    df["time"] = pd.to_datetime(df["time"])

    print(
        Path(file).name,
        "|",
        df["time"].min(),
        "|",
        df["time"].max(),
        "|",
        len(df)
    )

open-meteo-22.46N88.32E10m_VIVEKANANDA COLLEGE FOR WOMEN, BARISHA.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.46N88.32E3m_Sarsuna College.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.46N88.41E3m_Avidipta Housing Complex.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.46N88.51E3m_Leather Complex.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.53N88.32E11m_Lorreto College.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.53N88.32E12m_Maulana Azad Collage.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.53N88.32E7m_Ballygunge Campus, C.U.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.53N88.41E0m_Urbana Housing Complex.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.53N88.41E3m_Fortis Hospital.xlsx | 2022-01-01 00:00:00 | 2026-05-30 23:00:00 | 38664
open-meteo-22.53N88.41E5m_Dhapa Lock Pumping Station.xl

# Missing Value Analysis

In [8]:
for file in weather_files:

    df = pd.read_excel(file)

    df = clean_weather_columns(df)

    missing = (
        df.isnull()
          .sum()
          .sum()
    )

    print(
        Path(file).name,
        "-> Missing:",
        missing
    )

open-meteo-22.46N88.32E10m_VIVEKANANDA COLLEGE FOR WOMEN, BARISHA.xlsx -> Missing: 0
open-meteo-22.46N88.32E3m_Sarsuna College.xlsx -> Missing: 0
open-meteo-22.46N88.41E3m_Avidipta Housing Complex.xlsx -> Missing: 0
open-meteo-22.46N88.51E3m_Leather Complex.xlsx -> Missing: 0
open-meteo-22.53N88.32E11m_Lorreto College.xlsx -> Missing: 0
open-meteo-22.53N88.32E12m_Maulana Azad Collage.xlsx -> Missing: 0
open-meteo-22.53N88.32E7m_Ballygunge Campus, C.U.xlsx -> Missing: 0
open-meteo-22.53N88.41E0m_Urbana Housing Complex.xlsx -> Missing: 0
open-meteo-22.53N88.41E3m_Fortis Hospital.xlsx -> Missing: 0
open-meteo-22.53N88.41E5m_Dhapa Lock Pumping Station.xlsx -> Missing: 0
open-meteo-22.53N88.41E8m_Flora Fountain.xlsx -> Missing: 0
open-meteo-22.53N88.41E9m_Lady Brabourne College.xlsx -> Missing: 0
open-meteo-22.60N88.32E10m_Presidency University.xlsx -> Missing: 0
open-meteo-22.60N88.41E10m_Sarojini Naidu College for Women.xlsx -> Missing: 0
open-meteo-22.60N88.41E12m_Bethune College.xlsx ->

# Duplicate Timestamp Check

In [9]:
for file in weather_files:

    df = pd.read_excel(file)

    df = clean_weather_columns(df)

    dup = (
        df["time"]
        .duplicated()
        .sum()
    )

    print(
        Path(file).name,
        "Duplicates:",
        dup
    )

open-meteo-22.46N88.32E10m_VIVEKANANDA COLLEGE FOR WOMEN, BARISHA.xlsx Duplicates: 0
open-meteo-22.46N88.32E3m_Sarsuna College.xlsx Duplicates: 0
open-meteo-22.46N88.41E3m_Avidipta Housing Complex.xlsx Duplicates: 0
open-meteo-22.46N88.51E3m_Leather Complex.xlsx Duplicates: 0
open-meteo-22.53N88.32E11m_Lorreto College.xlsx Duplicates: 0
open-meteo-22.53N88.32E12m_Maulana Azad Collage.xlsx Duplicates: 0
open-meteo-22.53N88.32E7m_Ballygunge Campus, C.U.xlsx Duplicates: 0
open-meteo-22.53N88.41E0m_Urbana Housing Complex.xlsx Duplicates: 0
open-meteo-22.53N88.41E3m_Fortis Hospital.xlsx Duplicates: 0
open-meteo-22.53N88.41E5m_Dhapa Lock Pumping Station.xlsx Duplicates: 0
open-meteo-22.53N88.41E8m_Flora Fountain.xlsx Duplicates: 0
open-meteo-22.53N88.41E9m_Lady Brabourne College.xlsx Duplicates: 0
open-meteo-22.60N88.32E10m_Presidency University.xlsx Duplicates: 0
open-meteo-22.60N88.41E10m_Sarojini Naidu College for Women.xlsx Duplicates: 0
open-meteo-22.60N88.41E12m_Bethune College.xlsx Du

# Metadata Validation

In [10]:
print("=" * 50)

print("Metadata Shape:", metadata.shape)

print(
    "Unique Stations:",
    metadata["station_id"].nunique()
)

print(
    "Unique Names:",
    metadata["station_name"].nunique()
)

print("=" * 50)

metadata[
    [
        "station_id",
        "station_name",
        "latitude",
        "longitude"
    ]
].head()

Metadata Shape: (17, 12)
Unique Stations: 17
Unique Names: 17


,station_id,station_name,latitude,longitude
0,WB001,Avidipta Housing Complex,22.493217,88.398310
1,WB002,"Ballygunge Campus, C.U",22.526701,88.363210
2,WB003,Bethune College,22.588510,88.367807
3,WB004,Dhapa Lock Pumping Station,22.558027,88.409980
4,WB005,"East Calcutta Girls College, Lake Town",22.601583,88.404556


# Build Weather Master Dataset

In [11]:
weather_master = []

station_lookup = {
    row["station_name"].strip(): row["station_id"]
    for _, row in metadata.iterrows()
}

for file in weather_files:

    filename = Path(file).stem

    station_name = filename.split("_", 1)[1].strip()

    if station_name not in station_lookup:
        print(
            f"Metadata Missing: {station_name}"
        )
        continue

    station_id = station_lookup[station_name]

    df = pd.read_excel(file)

    df = clean_weather_columns(df)

    df["datetime"] = pd.to_datetime(
        df["time"]
    )

    df["station_id"] = station_id

    weather_master.append(df)

print(
    f"Loaded Weather Stations: {len(weather_master)}"
)

Loaded Weather Stations: 17


# Concatenate Weather Dataset

In [12]:
weather_master = pd.concat(
    weather_master,
    ignore_index=True
)

print(weather_master.shape)

(657288, 12)


# Remove Original Time Column

In [13]:
weather_master = weather_master.drop(
    columns=["time"],
    errors="ignore"
)

# Sort Dataset

In [14]:
weather_master = (
    weather_master
    .sort_values(
        ["station_id", "datetime"]
    )
    .reset_index(drop=True)
)

# Weather Master Validation

In [15]:
print("=" * 60)

print("Rows:")
print(len(weather_master))

print("\nStations:")
print(weather_master["station_id"].nunique())

print("\nMissing Values:")
print(weather_master.isnull().sum().sum())

print("\nDuplicate Station-Datetime Pairs:")
print(
    weather_master.duplicated(
        subset=["station_id", "datetime"]
    ).sum()
)

print("=" * 60)

Rows:
657288

Stations:
17

Missing Values:
0

Duplicate Station-Datetime Pairs:
0


# Station Coverage Check

In [16]:
weather_master.groupby(
    "station_id"
).size().sort_index()

station_id
WB001    38664
WB002    38664
WB003    38664
WB004    38664
WB005    38664
WB006    38664
WB007    38664
WB008    38664
WB009    38664
WB010    38664
WB011    38664
WB012    38664
WB013    38664
WB014    38664
WB015    38664
WB016    38664
WB017    38664
dtype: int64

# Save Weather Master

In [17]:
weather_master.to_parquet(
    "../../data/processed/openmeteo_weather_master.parquet",
    index=False
)

print(
    "Saved: openmeteo_weather_master.parquet"
)

Saved: openmeteo_weather_master.parquet


# Reload Datasets

In [18]:
wbpcb = pd.read_parquet(
    "../../data/processed/wbpcb_master.parquet"
)

weather_master = pd.read_parquet(
    "../../data/processed/openmeteo_weather_master.parquet"
)

# Pre-Merge Validation

In [19]:
print("WBPCB Stations:",
      wbpcb["station_id"].nunique())

print("Weather Stations:",
      weather_master["station_id"].nunique())

print("WBPCB Datetime:",
      wbpcb["datetime"].dtype)

print("Weather Datetime:",
      weather_master["datetime"].dtype)

WBPCB Stations: 17
Weather Stations: 17
WBPCB Datetime: datetime64[us]
Weather Datetime: datetime64[us]


In [28]:
wbpcb["station_id"] = (
    wbpcb["station_id"]
    .astype(str)
    .str.strip()
)

weather_master["station_id"] = (
    weather_master["station_id"]
    .astype(str)
    .str.strip()
)

In [30]:
print(
    wbpcb["station_id"].nunique(),
    weather_master["station_id"].nunique()
)

print(
    wbpcb["station_id"].sort_values().unique()[:5]
)

print(
    weather_master["station_id"].sort_values().unique()[:5]
)

17 17
<ArrowStringArray>
['WB001', 'WB002', 'WB003', 'WB004', 'WB005']
Length: 5, dtype: str
<ArrowStringArray>
['WB001', 'WB002', 'WB003', 'WB004', 'WB005']
Length: 5, dtype: str


# Merge AQI + Weather

In [31]:
environmental_master = wbpcb.merge(
    weather_master,
    on=["station_id", "datetime"],
    how="left"
)

# Post-Merge Validation

In [32]:
print(environmental_master.shape)

environmental_master[
    [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m"
    ]
].isnull().sum()

(508116, 24)


temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
dtype: int64

# Drop Broken WBPCB Wind Features

In [33]:
environmental_master = (
    environmental_master
    .drop(
        columns=[
            "wind_direction",
            "wind_speed_avg",
            "wind_speed_min",
            "wind_speed_max"
        ]
    )
)

# Save Final Curated Dataset

In [34]:
environmental_master.to_parquet(
    "../../data/curated/environmental_master.parquet",
    index=False
)

print(
    "Saved: environmental_master.parquet"
)

Saved: environmental_master.parquet


# Final Dataset Validation

In [35]:
print("=" * 60)

print("Rows:",
      len(environmental_master))

print("Stations:",
      environmental_master["station_id"].nunique())

print("Columns:",
      len(environmental_master.columns))

print("Date Range:")

print(environmental_master["datetime"].min())

print(environmental_master["datetime"].max())

print("=" * 60)

Rows: 508116
Stations: 17
Columns: 20
Date Range:
2022-01-01 21:00:00
2026-05-30 21:00:00
